# Stage 01 - Data Ingestion: Ingestion and Processing from Relational SQL Databases
### Lecture 17: Relational-to-Document Serialization for RAG

This notebook implements database extraction:
1. Connecting to SQLite databases using Python `sqlite3`.
2. Inspecting relational schema tables and foreign keys.
3. Serializing SQL records into narrative documents with metadata.
4. Ensuring atomic extraction and clean database connection teardown.


In [1]:
# Step 1: Environment Setup and SQLite Connection
import sqlite3
from pathlib import Path
from langchain_core.documents import Document

repo_root = Path.cwd()
while not (repo_root / "data").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

db_path = repo_root / "data" / "databases" / "company.db"
print(f"[CONFIG] SQLite Database: {db_path} (Exists: {db_path.exists()})")
assert db_path.exists(), f"Database {db_path} must exist."

# Connect to database and inspect tables
conn = sqlite3.connect(str(db_path))
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = [row[0] for row in cursor.fetchall()]
print(f"[INFO] Discovered Tables in DB: {tables}")
assert len(tables) > 0, "Database must contain tables."


[CONFIG] SQLite Database: D:\Code\AI\rag-mastery\data\databases\company.db (Exists: True)
[INFO] Discovered Tables in DB: ['employees', 'projects']


In [2]:
# Step 2: Relational-to-Document Transformation
sql_documents = []

for table_name in tables:
    cursor.execute(f"PRAGMA table_info({table_name});")
    columns = [col[1] for col in cursor.fetchall()]
    
    cursor.execute(f"SELECT * FROM {table_name};")
    rows = cursor.fetchall()
    print(f"[INFO] Table '{table_name}': {len(rows)} records found (Columns: {columns})")
    
    for row in rows:
        row_dict = dict(zip(columns, row))
        narrative_str = f"Record in {table_name}: " + ", ".join([f"{k} is {v}" for k, v in row_dict.items()])
        sql_documents.append(
            Document(
                page_content=narrative_str,
                metadata={"source": db_path.name, "table": table_name, **{k: str(v) for k, v in row_dict.items() if k.endswith('_id') or k == 'id'}}
            )
        )

conn.close()
print(f"[INFO] Total generated SQL documents: {len(sql_documents)}")
for d in sql_documents[:2]:
    print(f"  [{d.metadata}]: {d.page_content}")

# Assertions
assert len(sql_documents) > 0, "SQL serialization must produce non-empty documents."
print("[SUCCESS] SQL database extraction and relational serialization verified.")


[INFO] Table 'employees': 4 records found (Columns: ['id', 'name', 'role', 'department', 'salary'])
[INFO] Table 'projects': 4 records found (Columns: ['id', 'name', 'status', 'budget', 'lead_id'])
[INFO] Total generated SQL documents: 8
  [{'source': 'company.db', 'table': 'employees', 'id': '1'}]: Record in employees: id is 1, name is John Doe, role is Senior Developer, department is Engineering, salary is 95000.0
  [{'source': 'company.db', 'table': 'employees', 'id': '2'}]: Record in employees: id is 2, name is Jane Smith, role is Lead Data Scientist, department is Data Science, salary is 115000.0
[SUCCESS] SQL database extraction and relational serialization verified.
